<a href="https://colab.research.google.com/github/nawrin1/First_Program_SM/blob/main/L9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Imports & Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## 2. Data Loading


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/datasets/cybersecurity_intrusion_data.csv')
print(f"Shape: {df.shape}")
df.head()

## 3. Data Inspection


In [ ]:
missing = df.isnull().sum()

print("Missing values per column:\n")
print(missing[missing > 0])

print("\nAttack Distribution:\n")
print(df['attack_detected'].value_counts().sort_index())

print("\nClass Mapping:")
print("0 = Normal")
print("1 = Attack")

## 4. Preprocessing & Train-Test Split


In [ ]:
df.drop(columns=['session_id'], inplace=True)
df = pd.get_dummies(
    df,
    columns=['protocol_type', 'encryption_used', 'browser_type'],
    drop_first=True
)

df.fillna(df.median(numeric_only=True), inplace=True)

X = df.drop(columns=['attack_detected'])
y = df['attack_detected']


X_scaled = StandardScaler().fit_transform(X)


X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=20
)
print(f"\nTrain: {X_train.shape[0]} samples")
print(f"Test : {X_test.shape[0]} samples")


## 5. Exploratory Data Analysis (EDA)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = ['Normal', 'Attack']


df['attack_detected'].value_counts().plot.bar(
    ax=axes[0],
    color=['#2ecc71', '#e74c3c']
)
axes[0].set_xticklabels(labels, rotation=0)
axes[0].set_title('Attack Distribution')
axes[0].set_ylabel('Count')


df.corr()['attack_detected'] \
    .drop('attack_detected') \
    .abs() \
    .sort_values(ascending=False) \
    .head(10) \
    .plot.barh(ax=axes[1], color='#3498db')

axes[1].set_title('Top-10 Features Correlated with Attack')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Model Training & Baseline Results


In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=20),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Logistic Reg': LogisticRegression(max_iter=1000, random_state=20)
}

results = {}
predictions = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    yp = model.predict(X_test)

    predictions[name] = yp

    results[name] = {
        'Accuracy': accuracy_score(y_test, yp),
        'Precision': precision_score(y_test, yp, average='weighted'),
        'Recall': recall_score(y_test, yp, average='weighted'),
        'F1-Score': f1_score(y_test, yp, average='weighted')
    }
    print(f"\n{name}")
    print(f"Accuracy  = {results[name]['Accuracy']:.4f}")
    print(f"Precision = {results[name]['Precision']:.4f}")
    print(f"Recall    = {results[name]['Recall']:.4f}")
    print(f"F1-Score  = {results[name]['F1-Score']:.4f}")

## 7. Decision Tree ? Depth Tuning


In [ ]:
dt_results = []

for d in [3, 5, 10, 15, 20, None]:

    dt = DecisionTreeClassifier(
        max_depth=d,
        random_state=20
    )

    dt.fit(X_train, y_train)

    yp = dt.predict(X_test)
    dt_results.append({
        'max_depth': str(d),
        'Accuracy': accuracy_score(y_test, yp),
        'Precision': precision_score(y_test, yp, average='weighted'),
        'Recall': recall_score(y_test, yp, average='weighted'),
        'F1-Score': f1_score(y_test, yp, average='weighted')
    })

dt_df = pd.DataFrame(dt_results).set_index('max_depth')
print("\nDecision Tree Depth Results:\n")
print(dt_df.round(4))

# Plot Results
dt_df.plot(
    kind='bar',
    figsize=(10, 5),
    colormap='viridis',
    edgecolor='black'
)
plt.title('Decision Tree vs Max Depth')
plt.ylabel('Score')
plt.ylim(0.5, 1.02)

plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 8. Classification Reports


In [ ]:
target_names = ['Normal', 'Attack']

for name, yp in predictions.items():

    print(f"\n{'='*55}")
    print(name)
    print(f"{'='*55}")
    print(
        classification_report(
            y_test,
            yp,
            target_names=target_names,
            digits=4
        )
    )


## 9. Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, yp) in zip(axes, predictions.items()):

    sns.heatmap(
        confusion_matrix(y_test, yp),
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=ax,
        xticklabels=target_names,
        yticklabels=target_names
    )
    ax.set_title(name)

    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()


## 10. Model Comparison


In [ ]:
res_df = pd.DataFrame(results).T

res_df.plot(
    kind='bar',
    figsize=(10, 5),
    colormap='Set2',
    edgecolor='black'
)

plt.title('Model Performance Comparison')

plt.ylabel('Score')

plt.ylim(0.5, 1.02)
plt.xticks(rotation=0)

plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

print("\nSummary:\n")
print(res_df.round(4))

## 11. Assignment

**Dataset:** [Cybersecurity Intrusion Detection Dataset](https://www.kaggle.com/datasets/dnkumars/cybersecurity-intrusion-detection-dataset)

Upload the dataset to your Google Drive under `datasets/` and complete the following tasks using the **same workflow** as this notebook.

| Task | What to do |
|------|------------|
| **1. Load & Inspect** | Load CSV, print shape, missing values, and class distribution |
| **2. Preprocess** | Drop irrelevant columns, fill missing values with median, scale features, split 80/20 |
| **3. EDA** | Bar chart of class distribution + top-10 feature correlations |
| **4. Train Models** | Decision Tree, KNN (k=5), Logistic Regression ? report Accuracy, Precision, Recall, F1 |
| **5. Depth Tuning** | Decision Tree with max_depth = [3, 5, 10, 15, 20, None] ? table + bar chart |
| **6. Evaluate** | Classification reports + 1?3 confusion matrix grid + model comparison bar chart |

**Submit:** Single `.ipynb` with all cells executed and clean output.